In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A2_generation/6_predicting_notes.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Rudiments of Machine Learning: Learning a Continuous Function

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 20.05.2026
+ **Authors:** [Dr. Benedikt Zönnchen](https://bzoennchen.github.io/Pages/)

In [ ]:
#@title Setup: install required Python packages

%pip install otter-grader==5.5.0
%pip install numpy
%pip install scikit-learn
%pip install torch
%pip install torchview
%pip install graphviz
%pip install matplotlib
%pip install seaborn

In [ ]:
#@title Setup: download assignment files (run this cell)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # download test files
    import requests, os
    os.makedirs('tests', exist_ok=True)
    folders = ['tests']
    link = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A2_generation"

    def download(entry, dest):
        if entry.get('type') != 'file' or not entry.get('download_url'):
            return
        r = requests.get(entry['download_url'])
        r.raise_for_status()
        with open(dest, 'wb') as out:
            out.write(r.content)

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        for f in requests.get(f"{link}/{folder}").json():
            download(f, f"{folder}/{f['name']}")

    for f in requests.get(link).json():
        if f['name'].endswith('.py'):
            download(f, f['name'])

    # Initialize Otter
    import otter
    grader = otter.Notebook(colab=True)
else:
    import otter
    grader = otter.Notebook('6_predicting_notes.ipynb')

## 15 Predicting Notes in a p5.js Sketch (again)

In this notebook we will repeat what we learned in block 1. We again look at two types of *supervised learning* tasks:

+ regression and
+ classification.

We solve the two problems from [C) Hands-on : train your first model (classification and regression)](https://aica-wavelab.github.io/tech-crash-course/content/tutorials/1_intro_iml/#c-hands-on--train-your-first-model-classification-and-regression) but this time we define our own neural network using `Python` code.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from torchview import draw_graph

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler

The following function `plot_frequncies` helps us visualizing our progress.

In [ ]:
def plot_sketch(X, y, hue_norm=None):
    """Plot 2D points colored by their frequency values.

    Args:
        X: Iterable of (x, y) coordinate pairs.
        y: Our labels, that is, frequency (float) or a class (int) value for each point, used as the color hue.
        hue_norm: Optional (vmin, vmax) tuple to fix the color scale,
            so equal frequencies map to equal colors across plots.

    Returns:
        None. Displays the plot via plt.show().
    """

    if isinstance(X, torch.Tensor):
      X = X.numpy()

    if isinstance(X, torch.Tensor):
      y = y.numpy()

    x_coord = [c[0] for c in X]
    y_coord = [c[1] for c in X]

    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=x_coord, y=y_coord, hue=y, palette="viridis", s=120, edgecolor="k", hue_norm=hue_norm)
    plt.show()

To use it put the coordinates in `X` and the frequencies in `y` like this:

In [ ]:
X = [[0,0], [40, 40], [30, 10]]
y = [300, 400, 100]
plot_sketch(X,y)

### 15.1 Predicting Frequencies - Regression (again)

#### 15.1.1 Data Collection

---

🖍 **Exercise 15.1:** 

1. Go to [regression-sketch](https://editor.p5js.org/BZoennchen/sketches/lHQAFZhCi) and create your own training data set by executing the `p5.js` sketch.
2. Download your data by using the `s`-key.
3. But the data into the `data` folder (create this folder if it doesn't exist)
4. Chage the `filename` to the name of your file.
5. Execute the cell below to load the data into this notebook.

---

In [ ]:
import json
import os

filename = '2026-5-14_16-37-23.json'

with open('data/'+filename) as f:
  data_json = json.loads(f.read())

data_json

To make it easy for you to transform this Json into a tupel of proper `torch.tensor` that represent our input data and the label you can use `json_to_tensor`:

In [ ]:
def json_to_lists(data_json, label_key):

  X_list = []
  y_list = []

  for datapoint in data_json['data']:
    xcoord, ycoord = datapoint['xs']['x'], datapoint['xs']['y']
    X_list.append([xcoord, ycoord])
    y_list.append(datapoint['ys'][label_key])

  return X_list, y_list

In [ ]:
X_list, y_list = json_to_lists(data_json, label_key='frequency')

X = torch.tensor(X_list, dtype=torch.float32)
y = torch.tensor(y_list, dtype=torch.float32)

print(f'input: {X[:3]}')
print(f'output: {y[:3]}')

Let us plot the data.

In [ ]:
plot_sketch(X, y)

---

🖍 **Exercise 15.2:** Define a `torch.tensor` `W` such that `X @ W` works!

---

In [ ]:
print(X.shape) # 50 x 2
print(X.dtype)

In [ ]:
W = ...
A = X @ W

In [ ]:
grader.check("q152")

---

🖍 **Exercise 15.3:** Define a `torch.tensor` `S` such that `X @ S` swaps all x- and y-coordinates. The shape of the resulting tensor is the same as `X`!

---

In [ ]:
S = ...

In [ ]:
grader.check("q153")

<!-- BEGIN QUESTION -->

#### 15.1.2 Model Definition

---

🖍 **Exercise 15.4:** 

1. Again go to [regression-sketch](https://editor.p5js.org/BZoennchen/sketches/lHQAFZhCi)
2. Figure out how the architecture of the model we train there, i.e. the layers / number of parameters of the artificial neural network.
3. Build this model in using `torch`.

---

In [ ]:
class FeedForwardRegression(nn.Module):
  def __init__(self):
    super(FeedForwardRegression, self).__init__()
    ...

  def forward(self, x):
    ...
    return out

<!-- END QUESTION -->

#### 15.1.3 Data Preparation

`X` contains all the coordinates and `y` the frequencies. `X` is the input for the neural network where as `y` what it should predict.
It is always a good idea to normalize or standardize the input `X` and sometimes it might be useful to do the same for `y`.

---

🖍 **Exercise 15.5:** prepare your data for training.

---

Usually it is enough to standardize the input. However, if numbers get to large, e.g. the loss explodes, we might run into numerical problems. Therefore, we also scale down the output (frequencies).

In [ ]:
# this is required if you execute this cell multiple times
X = torch.tensor(X_list, dtype=torch.float32)
y = torch.tensor(y_list, dtype=torch.float32)

# scale input
input_scaler = StandardScaler()
X = torch.tensor(input_scaler.fit_transform(X), dtype=torch.float32)
print(X[:5])

# this is torch specific
y = y.reshape(-1, 1)

# we also scale output because we observed huge losses thus gradients
output_scaler = StandardScaler()
y = torch.tensor(output_scaler.fit_transform(y), dtype=torch.float32)

print(y[:5])

Let us create our model and use it.

---

🗣 **Remark:** Whenever, we use our model outside of training we do not need to compute gradients. Thus it is good practice to tell `torch` that it is ok to not compute them! That's what `torch.no_grad()` does.

---

In [ ]:
model = FeedForwardRegression()

In [ ]:
with torch.no_grad():
  model.eval()
  print(model(X[:3]))

Let us visualize the neural network.

In [ ]:
with torch.no_grad():
  model_graph = draw_graph(model, input_data=X)
model_graph.visual_graph

#### 15.1.4 Training

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 15.6:** Train your model! Try to use the same number of epochs compared to the p5.js sketch, that is `50`.

---

In [ ]:
model = FeedForwardRegression()
optimizer = optim.Adam(model.parameters(), lr=0.5)
criterion = nn.MSELoss()

epochs = ...
losses = []

model.train()

for epoch in range(epochs):

  # clear old gradients
  optimizer.zero_grad()
  
  # forward pass
  predictions = ...

  # compute loss
  loss = ...

  # backprop i.e. compute gradients
  loss.backward()

  # update: w <- w - lr * grad
  optimizer.step()

  losses.append(loss.item())

print(f'loss: {losses}')

<!-- END QUESTION -->

#### 15.1.5 Evaluation

Let's create some random coordinates to test what our model is capable of.

In [ ]:
n_predictions = 1000
new_coords = np.random.rand(n_predictions,2) * 600
scaled_coords = torch.tensor(input_scaler.transform(new_coords), dtype=torch.float32)

In [ ]:
model.eval()
with torch.no_grad():
  predictions = model(scaled_coords)
predictions[:3]

In [ ]:
norm = (min(predictions.min(), y.min()), max(predictions.max(), y.max()))

In [ ]:
plot_sketch(X=scaled_coords.numpy(), y=predictions.flatten().numpy(), hue_norm = norm)

Let's compare this with our training data.

In [ ]:
plot_sketch(X, y.flatten(), hue_norm = norm)

---

🗣 **Remark:** To scale back the input and / or output you can use the `input_scaler.inverse_transform` and `output_scaler.inverse_transform`.

---

In [ ]:
plot_sketch(input_scaler.inverse_transform(X.numpy()), output_scaler.inverse_transform(y.numpy()).flatten())

### 15.2 Predicting Pitch Classes - Classification (again)

#### 15.2.1 Data Collection

---

🖍 **Exercise 15.7:** 

1. Go to [classification-sketch](https://editor.p5js.org/BZoennchen/sketches/mHhBWnuoI) and create your own training data set by executing the `p5.js` sketch.
2. Download your data by using the `s`-key.
3. But the data into the `data` folder (create this folder if it doesn't exist)
4. Chage the `filename` to the name of your file.
5. Execute the cell below to load the data into this notebook.

---

In [ ]:
import json
import os

filename = '2026-5-14_17-46-46.json'

with open('data/'+filename) as f:
  data_json_c = json.loads(f.read())

X_list_c, y_list_c = json_to_lists(data_json_c, label_key = 'label')

In [ ]:
print(f'input: {X_list_c[:5]}')
print(f'output: {y_list_c[:5]}')

#### 15.2.2 Model Definition

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 15.8:** 

1. Again go to [classification-sketch](https://editor.p5js.org/BZoennchen/sketches/mHhBWnuoI)
2. Figure out how the architecture of the model we train there, i.e. the layers / number of parameters of the artificial neural network.
3. Build this model in using `torch`.

---

In [ ]:
class FeedForwardClassification(nn.Module):
  def __init__(self):
    super(FeedForwardClassification, self).__init__()
    ...
  
  def forward(self, x):
    ...
    return out

<!-- END QUESTION -->

#### 15.2.3 Data Preparation

A neural network can only handle numbers. Therefore, we have to translate our pitch classes into numbers.
One way to do this is to just count the number of classes and number them starting at 0.
We construct two dictionarys `intToClass` that maps the number representing the pitch class to the string representing this class.
`classToInt` does the reverse.

In [ ]:
classes = list(set(y_list_c))
n_classes = len(classes)
intToClass = {i: classes[i] for i in range(n_classes)}
classToInt = {classes[i] :i  for i in intToClass}

print(intToClass)
print(classToInt)

This time we only scale the input since the output is between 0 and 12.

In [ ]:
X_c = torch.tensor(X_list_c, dtype=torch.float32)
y_c = torch.tensor([classToInt[y] for y in y_list_c], dtype=torch.long)

standard_scaler = StandardScaler()
X_c = torch.tensor(standard_scaler.fit_transform(X_c), dtype=torch.float32)

In [ ]:
y_c

<!-- BEGIN QUESTION -->

#### 15.2.4 Training

---

🖍 **Exercise 15.9:** Train your model! Try to use the same number of epochs compared to the p5.js sketch, that is `50`.

---

In [ ]:
classification_model = FeedForwardClassification()
optimizer_c = optim.Adam(classification_model.parameters(), lr=0.5)
criterion_c = nn.CrossEntropyLoss()

epochs = 50
losses = []

classification_model.train()

for epoch in range(epochs):

  ...

print(f'loss: {losses}')

<!-- END QUESTION -->

#### 15.2.5 Evaluation

Let's create some random coordinates to test what our model is capable of.

In [ ]:
n_predictions = 1000
new_coords = np.random.rand(n_predictions,2) * 600
scaled_coords = torch.tensor(standard_scaler.transform(new_coords), dtype=torch.float32)

In [ ]:
classification_model.eval()
with torch.no_grad():
  logits = classification_model(scaled_coords)
  probs = torch.softmax(logits, dim=1) 
  predictions = logits.argmax(dim=1)
predictions[:10]

In [ ]:
norm = (min(predictions.min(), y.min()), max(predictions.max(), y.max()))

In [ ]:
plot_sketch(X=scaled_coords.numpy(), y=predictions.flatten().numpy(), hue_norm = norm)

In [ ]:
plot_sketch(X_c.numpy(), y_c.flatten().numpy(), hue_norm = norm)

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 15.10:** Use for your model only `nn.Linear` (e.g. no `nn.ReLU`) and try it out for a data set that is

1. Linear
2. Non-linear

What do you observer?

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 15.11:** What happens when you increase or decrease the size of your model?

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

